# OSL End-to-End Notebook (RSAC-GRU)

This notebook runs end-to-end on this repo in one place:
1. load project modules
2. train RSAC
3. evaluate checkpoints and plots

Backbone fixed in this notebook: `gru`

Repo quick review:
- Entry pipeline: `main.py`
- Training: `train.py` (`train(args)`)
- Evaluation: `eval.py` (`evaluate(args)`)
- Env/agent factory: `src/utils/factory.py`
- RSAC backbones (`gru`, `connectome2`): `src/models/networks.py`


In [ ]:
import os
from train import train
from eval import evaluate
from src.utils.config import build_parser
from src.utils.seed import set_global_seed

set_global_seed(42)
print("Imports ready")


In [ ]:
from pathlib import Path
import time

# Edit these values if needed.
OUT_DIR = "runs"
TOTAL_EPISODES = 500
EVAL_EPISODES = 50
SEED = 42
FORCE_CPU = False
REWARD_MODE = "bio"  # "bio" or "mechanical"

# RSAC + v4 defaults
ENV_ID = "OdorHold-v4"
AGENT_TYPE = "rsac"
BATCH_SIZE = 128
BUFFER_SIZE = 150000
LEARNING_STARTS = 5000
LOG_EVERY = 20

# Backbone settings for this notebook (fixed)
RSAC_ACTOR_BACKBONE = "gru"
SEQ_LEN = 16
CONNECTOME_HIDDEN = 180
CONNECTOME_STEPS = 4

run_stamp = time.strftime("%Y%m%d_%H%M%S")
RUN_NAME = f"gru_notebook_{run_stamp}"

print("RUN_NAME:", RUN_NAME)


In [ ]:
args = build_parser().parse_args([])

# Core run settings
args.env_id = ENV_ID
args.agent_type = AGENT_TYPE
args.out_dir = OUT_DIR
args.run_name = RUN_NAME
args.total_episodes = TOTAL_EPISODES
args.seed = SEED
args.force_cpu = FORCE_CPU

# Reward/env settings
args.reward_mode = REWARD_MODE

# Training settings
args.batch_size = BATCH_SIZE
args.buffer_size = BUFFER_SIZE
args.learning_starts = LEARNING_STARTS
args.log_every = LOG_EVERY

# RSAC actor backbone settings
args.rsac_actor_backbone = RSAC_ACTOR_BACKBONE
args.seq_len = SEQ_LEN
args.connectome_hidden = CONNECTOME_HIDDEN
args.connectome_steps = CONNECTOME_STEPS

print("Training config:")
for k in [
    "env_id", "agent_type", "total_episodes", "seed", "reward_mode",
    "rsac_actor_backbone", "seq_len", "connectome_hidden", "connectome_steps",
    "batch_size", "learning_starts", "run_name"
]:
    print(f"  {k}: {getattr(args, k)}")

train_result = train(args)
run_dir = train_result["run_dir"]
print("\nTraining finished. run_dir:", run_dir)


In [ ]:
eval_args = build_parser().parse_args([])
eval_args.run_dir = run_dir
eval_args.ckpt = None  # auto-pick best.pt (or final.pt)
eval_args.episodes = EVAL_EPISODES
eval_args.seed_base = 20000
eval_args.seed = SEED
eval_args.force_cpu = FORCE_CPU
eval_args.save_gif = True
eval_args.plot_milestones = True

# Optional explicit override (kept same as training backbone)
eval_args.rsac_actor_backbone = RSAC_ACTOR_BACKBONE
eval_args.connectome_hidden = CONNECTOME_HIDDEN
eval_args.connectome_steps = CONNECTOME_STEPS

print("Evaluation started...")
evaluate(eval_args)
print("Evaluation finished.")


In [ ]:
from pathlib import Path

run_path = Path(run_dir)
print("\nSaved files:")
for p in sorted(run_path.glob("*")):
    print(" -", p)

ckpt_dir = run_path / "checkpoints"
plot_dir = run_path / "plots"

print("\nCheckpoints:")
if ckpt_dir.exists():
    for p in sorted(ckpt_dir.glob("*.pt")):
        print(" -", p.name)
else:
    print(" - (missing)")

print("\nPlots:")
if plot_dir.exists():
    for p in sorted(plot_dir.glob("*")):
        print(" -", p.name)
else:
    print(" - (missing)")
